# Lab: Gaussian Elimination with Partial Pivoting

---

## Overview

This lab implements and runs the **Gaussian Elimination** algorithm written in C++ (`gaussian_elimination.cpp`).  
You will compile the program, run it on several test cases, and analyze its behavior.

## Learning Objectives

By the end of this lab, you will be able to:

1. Explain the three phases of Gaussian elimination: **pivoting**, **forward elimination**, and **back substitution**
2. Describe how **partial pivoting** improves numerical stability
3. Interpret the **infinity-norm residual** $\|Ax - b\|_\infty$ as a correctness metric
4. Observe the effect of **ill-conditioned matrices** (Hilbert matrix) on solution accuracy
5. Compile and run a C++ linear-algebra program from a Jupyter notebook

## 0. Check CPU & Supported SIMD Extensions

Before choosing an SIMD instruction set, query the server's CPU capabilities.

## 1. View the Source File

Let's first inspect the C++ source code before compiling it.

In [ ]:
with open("gaussian_elimination.cpp", "r", encoding="utf-8") as f:
    print(f.read())

## 2. Algorithm Overview

The program solves the linear system $Ax = b$ using three phases:

### Phase 1 — Partial Pivoting (per column $k$)

1. Find $p = \arg\max_{i \geq k} |a_{ik}|$ — the row with the largest absolute value in column $k$
2. If $|a_{pk}| < \varepsilon$ ($\varepsilon = 10^{-9}$) → matrix is singular, abort
3. Swap row $k$ with row $p$

### Phase 2 — Forward Elimination (per column $k$, rows $i > k$)

$$
m_{ik} = \frac{a_{ik}}{a_{kk}}, \qquad
a_{ij} \leftarrow a_{ij} - m_{ik} \cdot a_{kj}, \quad j = k, \ldots, n
$$

After all columns, $[A \mid b]$ is in **upper-triangular** form.

### Phase 3 — Back Substitution

$$
x_i = \frac{b_i - \displaystyle\sum_{j=i+1}^{n-1} a_{ij}\, x_j}{a_{ii}}, \quad i = n-1, \ldots, 0
$$

### Exercise: Thread Index Table

Fill in the global index for each step of forward elimination ($n = 4$):

| Column $k$ | Pivot row | Rows eliminated below |
|:-----------|:----------|:----------------------|
| 0 | ______ | 1, 2, 3 |
| 1 | ______ | 2, 3 |
| 2 | ______ | 3 |
| 3 | ______ | — |

## 3. Compile the Program

We compile `gaussian_elimination.cpp` with `g++` using `-O2` optimization and C++17 standard.

In [ ]:
import subprocess, os, sys

SRC = "gaussian_elimination.cpp"
EXE = "gaussian_elimination" + (".exe" if sys.platform == "win32" else "")

result = subprocess.run(
    ["g++", "-O2", "-std=c++17", "-o", EXE, SRC],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(f"[OK] Compiled: {SRC} -> {EXE}")
else:
    print("[FAILED] Compilation error:")
    print(result.stderr)

## 4. Run — Example 1: 4×4 System

The program's built-in Example 1 uses the following augmented matrix (exact solution: $x = [1,2,3,4]^T$):

$$
[A \mid b] =
\begin{bmatrix}
4 & 1 & 2 & 3 & \mid & 24 \\
3 & 4 & 1 & 2 & \mid & 22 \\
2 & 3 & 4 & 1 & \mid & 24 \\
1 & 2 & 3 & 4 & \mid & 30
\end{bmatrix}
$$

We pass `0` on stdin to skip the interactive custom-input section.

In [ ]:
result = subprocess.run(
    [os.path.join(".", EXE)],
    input="0\n",
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

## 5. Run — Example 2: Hilbert Matrix (Ill-Conditioned)

The **Hilbert matrix** $H_{ij} = \dfrac{1}{i+j+1}$ is a classic ill-conditioned matrix.  
Its condition number grows rapidly with $n$, amplifying round-off errors.

| $n$ | $\kappa(H_n)$ (approx.) |
|:----|------------------------:|
| 3 | $5.2 \times 10^{2}$ |
| 5 | $4.8 \times 10^{5}$ |
| 8 | $1.5 \times 10^{10}$ |

The program constructs $b = H \cdot \mathbf{1}$, so the exact solution is $x = [1, 1, 1]^T$.

**Question**: Does the residual for the Hilbert matrix look different from Example 1? Why?

Your answer: ______

## 6. Run — Custom Input

Supply your own linear system via stdin.  
The input format is:
```
n
a[0][0] a[0][1] ... a[0][n-1] b[0]
a[1][0] ...                   b[1]
...
a[n-1][0] ...           b[n-1]
```

The cell below solves the 3×3 system:

$$
\begin{bmatrix} 2 & 1 & -1 \\ -3 & -1 & 2 \\ -2 & 1 & 2 \end{bmatrix}
\begin{bmatrix} x_0 \\ x_1 \\ x_2 \end{bmatrix}
=
\begin{bmatrix} 8 \\ -11 \\ -3 \end{bmatrix}
\quad \text{(exact solution: } x = [2, 3, -1]^T \text{)}
$$

In [ ]:
# Custom input: 3x3 system, exact solution x = [2, 3, -1]
custom_input = """\
3
2 1 -1 8
-3 -1 2 -11
-2 1 2 -3
"""

result = subprocess.run(
    [os.path.join(".", EXE)],
    input=custom_input,
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

## 7. Run — Singular Matrix Detection

When the matrix is singular (or near-singular), the program should detect a near-zero pivot and report an error.

We test with a rank-deficient 3×3 matrix (row 2 = row 0 + row 1).

In [ ]:
# Singular matrix: row2 = row0 + row1 (rank-deficient)
singular_input = """\
3
1 2 3 6
4 5 6 15
5 7 9 21
"""

result = subprocess.run(
    [os.path.join(".", EXE)],
    input=singular_input,
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

## 8. Verify Results with Python (numpy)

Use `numpy.linalg.solve` as a reference to verify the C++ output is correct.

In [ ]:
import numpy as np

def check_system(A, b, label=""):
    x_np = np.linalg.solve(A, b)
    residual = np.max(np.abs(A @ x_np - b))
    print(f"{'[' + label + '] ' if label else ''}numpy solution: x = {np.round(x_np, 6).tolist()}")
    print(f"  Residual ||Ax-b||_inf = {residual:.3e}\n")

# Example 1 — 4x4
A1 = np.array([[4,1,2,3],[3,4,1,2],[2,3,4,1],[1,2,3,4]], dtype=float)
b1 = np.array([24, 22, 24, 30], dtype=float)
check_system(A1, b1, "Example 1 (expected x=[1,2,3,4])")

# Example 2 — Hilbert 3x3
n = 3
H = np.array([[1.0/(i+j+1) for j in range(n)] for i in range(n)])
b2 = H @ np.ones(n)
check_system(H, b2, "Hilbert 3x3 (expected x=[1,1,1])")

# Custom — 3x3
A3 = np.array([[2,1,-1],[-3,-1,2],[-2,1,2]], dtype=float)
b3 = np.array([8,-11,-3], dtype=float)
check_system(A3, b3, "Custom 3x3 (expected x=[2,3,-1])")

## 9. Exercise: Record Your Results

Fill in the table after running all experiments above:

| Test Case | n | Computed x | Residual $\|Ax-b\|_\infty$ | Correct? |
|:----------|:-:|:-----------|:--------------------------:|:--------:|
| Example 1 (4×4) | 4 | ______ | ______ | ______ |
| Hilbert 3×3 | 3 | ______ | ______ | ______ |
| Custom 3×3 | 3 | ______ | ______ | ______ |
| Singular 3×3 | 3 | N/A (error) | N/A | ✓ |

**Question 1**: For a well-conditioned system, what order of magnitude do you expect for the residual?

Your answer: ______

**Question 2**: For the Hilbert matrix, is the residual larger or smaller than for Example 1? Why?

Your answer: ______

**Question 3**: What does the program print when the pivot falls below $\varepsilon = 10^{-9}$?

Your answer: ______

## 10. Summary

### Algorithm Complexity

| Phase | Operations |
|:------|:-----------|
| Partial pivoting | $O(n^2)$ comparisons |
| Forward elimination | $O(n^3)$ multiply-add |
| Back substitution | $O(n^2)$ multiply-add |
| **Total** | $O(n^3)$ |

### Key Takeaways

| Concept | Detail |
|:--------|:-------|
| **Partial pivoting** | Swap rows so the largest-magnitude entry is on the diagonal → reduces round-off error |
| **Forward elimination** | Reduce $[A \mid b]$ to upper-triangular form |
| **Back substitution** | Recover $x$ row by row from bottom to top |
| **Residual $\|Ax-b\|_\infty$** | Near machine epsilon ($\sim 10^{-15}$) for well-conditioned systems |
| **Ill-conditioned matrices** | Large $\kappa(A)$ amplifies floating-point errors even with pivoting |

### Best Practices

| Guideline | Reason |
|:----------|:-------|
| Always use partial pivoting | Avoids division by near-zero values |
| Check pivot vs. EPS | Detect singular / rank-deficient systems early |
| Verify with residual norm | Quantifies actual accuracy of the computed solution |

---

## Appendix: Reference Files

| File | Description |
|:-----|:------------|
| `gaussian_elimination.cpp` | Main C++ source: pivoting, elimination, back-sub, residual check |
| `Makefile` | Build script (`make` to compile, `make run` to run) |

---

## SIMD Optimization: AVX2 Vectorized Gaussian Elimination

### Motivation

The inner elimination loop is the performance bottleneck:

```cpp
for (int k = col+1; k <= n; ++k)
    row[k] -= factor * pivot_row[k];   // n FMAs per row, O(n²) rows total
```

With **AVX2**, we process **4 `double`s per instruction** using `_mm256_fmadd_pd`, achieving up to **4× throughput** on this loop.

### Key Design Decisions

| Decision | Detail |
|:---------|:-------|
| **Flat aligned storage** | 1-D row-major array, 32-byte aligned → enables `_mm256_load_pd` |
| **Row stride = multiple of 4** | Guarantees every row starts on a 32-byte boundary |
| **FMA instruction** | `vfnmadd231pd`: `dst += -factor * src` in a single fused op |
| **Row swap via AVX2** | Swap entire rows with 256-bit loads/stores |
| **Scalar tail** | Handles remaining elements when `(n+1)` is not divisible by 4 |

### SIMD Elimination Loop (core)

```cpp
__m256d vfactor = _mm256_set1_pd(-factor);   // broadcast -factor to 4 lanes
int k = col + 1;
for (; k + 4 <= n + 1; k += 4) {
    __m256d vr = _mm256_loadu_pd(rr + k);        // load 4 doubles from target row
    __m256d vp = _mm256_loadu_pd(pivot_rp + k);  // load 4 doubles from pivot row
    vr = _mm256_fmadd_pd(vfactor, vp, vr);        // vr = vr + (-factor)*vp
    _mm256_storeu_pd(rr + k, vr);                 // store back
}
for (; k <= n; ++k)                               // scalar tail
    rr[k] -= factor * pivot_rp[k];
```

### Step 1: View the SIMD Source File

In [ ]:
with open("gaussian_elimination_simd.cpp", "r", encoding="utf-8") as f:
    print(f.read())

### Step 2: Compile with AVX2 + FMA Flags

In [ ]:
import subprocess, os, sys

SRC_SIMD = "gaussian_elimination_simd.cpp"
EXE_SIMD = "gaussian_elimination_simd" + (".exe" if sys.platform == "win32" else "")

result = subprocess.run(
    ["g++", "-O2", "-std=c++17", "-mavx2", "-mfma", "-o", EXE_SIMD, SRC_SIMD],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"[OK] Compiled: {SRC_SIMD} -> {EXE_SIMD}")
    print("Flags used: -O2 -std=c++17 -mavx2 -mfma")
else:
    print("[FAILED] Compilation error:")
    print(result.stderr)

### Step 3: Run SIMD Version — Built-in Examples

In [ ]:
result = subprocess.run(
    [os.path.join(".", EXE_SIMD)],
    input="0\n",
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

### Step 4: Performance Comparison — Scalar vs AVX2

We use `time` to compare the wall-clock time of both versions on a large random system ($n = 1000$).  
A Python helper generates the input matrix.

In [ ]:
import numpy as np
import time

def generate_input(n, seed=42):
    """Generate a diagonally dominant n×n system with a known solution."""
    rng = np.random.default_rng(seed)
    A = rng.random((n, n))
    np.fill_diagonal(A, A.diagonal() + n)   # diagonal dominance → non-singular
    x_exact = rng.random(n)
    b = A @ x_exact
    # Build augmented matrix string for stdin
    lines = [str(n)]
    for i in range(n):
        row = " ".join(f"{A[i,j]:.10f}" for j in range(n)) + f" {b[i]:.10f}"
        lines.append(row)
    return "\n".join(lines) + "\n"

N = 500   # system size (increase to 1000 on a fast server)
stdin_data = generate_input(N)
print(f"Generated {N}x{N} system ({len(stdin_data)//1024} KB of input)")

EXE_SCALAR = "gaussian_elimination" + (".exe" if sys.platform == "win32" else "")

def run_and_time(exe, stdin_data, label):
    t0 = time.perf_counter()
    r = subprocess.run(
        [os.path.join(".", exe)],
        input=stdin_data,
        capture_output=True, text=True, encoding="utf-8"
    )
    t1 = time.perf_counter()
    elapsed = (t1 - t0) * 1000
    if r.returncode != 0:
        print(f"[{label}] ERROR: {r.stderr[:200]}")
    else:
        # Extract residual line from output
        for line in r.stdout.splitlines():
            if "Residual" in line:
                print(f"[{label}]  {line.strip()}   |  Wall time: {elapsed:.1f} ms")
                break

run_and_time(EXE_SCALAR, stdin_data, "Scalar ")
run_and_time(EXE_SIMD,   stdin_data, "AVX2   ")

### Exercise: Record Your Results

Fill in after running the performance comparison above:

| Version | Compile flags | Wall time (ms) | Residual $\|Ax-b\|_\infty$ |
|:--------|:-------------|:--------------:|:--------------------------:|
| Scalar  | `-O2` | ______ | ______ |
| AVX2    | `-O2 -mavx2 -mfma` | ______ | ______ |
| Speedup | — | ______ × | — |